# Module 5.4: Capstone 2: Real-Time Anomaly & Hydraulic Leak Detector
### 🇵🇱 Projekt Zwieńczeniowy 2: Detektor Wycieków Oleju i Spadków Ciśnienia w Czasie Rzeczywistym

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m05_l04_realtime_anomaly_capstone2_leak_detector.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Build an end-to-end hydraulic leak detector using CUSUM and EWMA to detect micro-fissures and trigger automated safety shutdown alarms.  
> 🇵🇱 **Streszczenie:** Zbuduj kompletny detektor wycieków hydraulicznych wykorzystujący algorytmy CUSUM i EWMA do wykrywania mikropęknięć i wyzwalania alarmów.

- **Estimated Time:** 30 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Capstone 2: Real-Time Anomaly & Hydraulic Leak Detector
1. **Hydraulic Micro-Leak Hazards**: In 280-bar hydraulic circuits, pinhole hose leaks spray atomized oil mist, creating catastrophic fire hazards.
2. **Mass Balance Differential**: $Q_{diff} = Q_{in} - Q_{out}$. In closed circuits, $Q_{diff} pprox 0$. A persistent positive delta indicates lost fluid.
3. **Cumulative Sum (CUSUM)**:
   $$S_t = \max(0, S_{t-1} + (x_t - \mu_0 - k))$$
   Integrates tiny systematic shifts that never cross standard static thresholds.
4. **Safety State Machine**: `NORMAL` $	o$ `WATCH` (warning) $	o$ `ALARM` (technician page) $	o$ `EMERGENCY_SHUTDOWN` (automated solenoid dump valve).


In [ ]:
# 1. Simulate High-Pressure Hydraulic Circuit with Micro-Leak Injection
np.random.seed(42)
n_sec = 1200
t = np.arange(n_sec)

# Nominal operating pressure: 280 bar with normal pump ripple
p_nominal = 280.0 + 3.5 * np.sin(2 * np.pi * t / 15) + np.random.normal(0, 0.8, n_sec)

# Nominal mass balance differential: Mean = 0.0 L/min, Std = 0.15 L/min
flow_diff = np.random.normal(0.0, 0.15, n_sec)

# Inject pinhole hydraulic leak at step 700 (losing 0.45 L/min)
leak_start = 700
flow_diff[leak_start:] += 0.45
# Pressure decay following accumulator exhaustion
p_telemetry = p_nominal.copy()
p_telemetry[leak_start:] -= np.linspace(0, 35.0, n_sec - leak_start)

print(f"✓ Hydraulic simulation created: 280 bar nominal, leak initiated at t={leak_start}s.")


In [ ]:
# 2. Implement CUSUM Change-Point Detector & Multi-State Safety Machine
k_allowance = 0.10   # Slack parameter
h_watch = 4.0        # Warning threshold
h_shutdown = 12.0    # Emergency trip threshold

cusum_stat = np.zeros(n_sec)
alarm_states = []

for i in range(1, n_sec):
    cusum_stat[i] = max(0.0, cusum_stat[i-1] + (flow_diff[i] - k_allowance))
    if cusum_stat[i] >= h_shutdown:
        alarm_states.append("EMERGENCY_SHUTDOWN")
    elif cusum_stat[i] >= h_watch:
        alarm_states.append("WATCH")
    else:
        alarm_states.append("NORMAL")
alarm_states.insert(0, "NORMAL")

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# Pressure curve
axes[0].plot(t, p_telemetry, color='#0284c7', label='Accumulator Pressure [bar]')
axes[0].axvline(leak_start, color='#ef4444', linestyle='--', label='Micro-Leak Inception')
axes[0].set_ylabel("Pressure [bar]")
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Flow differential
axes[1].plot(t, flow_diff, color='#64748b', alpha=0.7, label='Mass Balance (Q_in - Q_out) [L/min]')
axes[1].axhline(0.0, color='gray', linestyle=':')
axes[1].set_ylabel("Flow Diff [L/min]")
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# CUSUM statistic and thresholds
axes[2].plot(t, cusum_stat, color='#ef4444', linewidth=2, label='CUSUM Error Accumulator')
axes[2].axhline(h_watch, color='#f59e0b', linestyle='--', linewidth=1.8, label=f'WATCH Level (h={h_watch})')
axes[2].axhline(h_shutdown, color='#ef4444', linestyle='-', linewidth=2.0, label=f'SHUTDOWN Trip (h={h_shutdown})')
axes[2].set_ylabel("CUSUM S_t")
axes[2].set_xlabel("Operational Seconds")
axes[2].legend(loc='upper left')
axes[2].grid(True, alpha=0.3)

plt.suptitle("Capstone 2: Real-Time Hydraulic Leak Detection & Safety Machine", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Simulate high-pressure hydraulic loop telemetry (inlet/outlet flow and pressure) *(🇵🇱 Zasymuluj telemetrię układu hydraulicznego wysokiego ciśnienia (przepływ i ciśnienie))*
- [ ] **Task 2:** Implement Cumulative Sum (CUSUM) change-point detector for micro-leak tracking *(🇵🇱 Zaimplementuj algorytm CUSUM do wykrywania mikroskopijnych wycieków magistrali)*
- [ ] **Task 3:** Deploy multi-state safety state machine (NORMAL, WATCH, ALARM, EMERGENCY_SHUTDOWN) *(🇵🇱 Wdróż wielostanowy automat bezpieczeństwa (NORMAL, WATCH, ALARM, EMERGENCY_SHUTDOWN))*


In [ ]:
# Exercise Challenge:
# Assert that WATCH was triggered within 25 seconds of leak inception
watch_triggered = np.where(cusum_stat >= h_watch)[0][0]
shutdown_triggered = np.where(cusum_stat >= h_shutdown)[0][0]

print(f"Leak Started at Step       : {leak_start}s")
print(f"WATCH Warning Triggered at : {watch_triggered}s (Latency: {watch_triggered - leak_start}s)")
print(f"SHUTDOWN Trip Triggered at : {shutdown_triggered}s")

assert watch_triggered < leak_start + 25, "CUSUM leak response latency was too slow!"
assert p_telemetry[shutdown_triggered] > 255.0, "Shutdown was delayed until pressure critically collapsed!"
print("✓ Capstone 2 hydraulic leak detector fully verified!")
